# Simulations (Section 4 and Appendix E)

This notebook reproduces the simulation figures in Section 4 and Appendix E.

Precomputed results are cached to `../data/full_results_all.pkl` so the figure cells can be run without recomputing. Set `rerun_full_algo=True` below to redo it.


In [ ]:
import sys, pickle, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import brentq

from scripts.sma import delta_pi_numba
from scripts.parameters import epsilon_target, J_I_from_epsilon
from scripts.experiment import run_minimal_experiment
from scripts.diagnostics import (evaluate_risk_profile, empirical_lipschitz_from_grid, local_lipschitz_from_grid, certify_worst_case_risk, lipschitz_constant)

root_folder = Path.cwd().parent
data_path = root_folder / 'data' / 'full_results_all.pkl'

## Run (or load) the main SMA simulation results

In [ ]:
rerun_full_algo = False # set True to rerun the full simulations from scratch (several hours)

if rerun_full_algo or not data_path.exists():
    m_values = np.linspace(1, 4, 16)
    full_results = []
    summary_rows = []

    for m in m_values:
        eps = epsilon_target(m)
        print(f"\n{'='*60}")
        print(f"Running m = {m:.1f}, epsilon = {eps:.4f}")
        print('='*60)

        res = run_minimal_experiment(
            m=m,
            epsilon=eps,
            n_samples_risk=1000,
            show_progress=True,
        )
        full_results.append(res)

        y_eval = np.array([0.5]) # example
        d_pi_vals = delta_pi_numba(y_eval, res['theta_grid'], res['avg_prior'])

        summary_rows.append({
            'm': m,
            'epsilon': eps,
            'I': res['I'],
            'J': res['J'],
            'd_pi_at_0.5': float(d_pi_vals[0]),
            'bayes_risk': res['bayes_risk'],
            'upper_bound': res['upper_bound'],
            'smd_runtime': res['smd_runtime'],
            'total_runtime': res['total_runtime']})

    with open(data_path, "wb") as f:
        pickle.dump(full_results, f)

    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))
else:
    with open(data_path, "rb") as f:
        full_results = pickle.load(f)
    print(f"Loaded {len(full_results)} results from {data_path}")
    print("m values:", [r['m'] for r in full_results])

full_results_all = sorted(full_results, key=lambda r: r['m'])

## Figure 1 - J (number of SMA iterations) and I (grid size) as functions of m

In [ ]:
# recommended number of iterations and grid points for m from 0.1 to 9
m_dense = np.linspace(0.1, 9, 500)

JI = np.array([J_I_from_epsilon(m, epsilon_target(m)) for m in m_dense], dtype=float)
J_vec, I_vec = JI[:, 0], JI[:, 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

ax1.plot(m_dense, J_vec / 1e6, color="navy", linewidth=1.8)
ax1.set_xlabel(r"$m$", fontsize=13)
ax1.set_ylabel(r"$J(m)$ (millions)", fontsize=13)
ax1.set_xlim(0, 9.1)
ax1.set_ylim(bottom=0)
ax1.set_axisbelow(True)
ax1.set_title(r"Number of iterations $J(m)$", fontsize=13)

ax2.plot(m_dense, I_vec, color="crimson", linewidth=1.8)
ax2.set_xlabel(r"$m$", fontsize=13)
ax2.set_ylabel(r"$I(m)$", fontsize=13)
ax2.set_xlim(0, 9.1)
ax2.set_ylim(bottom=0)
ax2.set_axisbelow(True)
ax2.set_title(r"Grid size $I(m)$", fontsize=13)

plt.tight_layout()
plt.show()

## Figure 2 - Computation time and $\epsilon(m)$


In [ ]:
# time of the actual algorithm run was recorded on a personal computer
results_le3 = sorted((r for r in full_results_all if r['m'] <= 3 +1e-9), key=lambda r: r['m'])
m_vals = [r['m'] for r in results_le3]
times = [r['smd_runtime'] for r in results_le3]
epsilons = [r['epsilon'] for r in results_le3]

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.plot(m_vals, times, '-', color='navy', linewidth=2, label='Time')
ax2.plot(m_vals, epsilons, '--', color='crimson', linewidth=2, label=r'$\epsilon(m)$')
ax1.set_xlabel(r'$m$', fontsize=13)
ax1.set_ylabel('Time (seconds)', fontsize=13, color='navy')
ax2.set_ylabel(r'$\epsilon$', fontsize=13, color='crimson')
ax1.tick_params(axis='y', labelcolor='navy')
ax2.tick_params(axis='y', labelcolor='crimson')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc='upper left')
plt.tight_layout()
plt.show()

## Figures 3, 4 (Section 4) and 15 (Appendix E) - Worst-case risks and improvement over linear minimax

In [ ]:
# cansider SMA-based and optimal linear estimator, and add two new estimators: truncated and clipped linear
lower_bounds = []
upper_bounds = []
upper_bounds_trunc = []
upper_bounds_lin_clip = []
upper_bounds_cert = []
upper_bounds_trunc_cert = []
upper_bounds_lin_clip_cert = []
m_vals = []
n_risk_mc = 1000
n_eval_cert = 200000 # fine grid for the certified bound only
noise = np.random.default_rng(123).standard_normal(n_risk_mc)
for res in tqdm(full_results_all, desc="Certified bound"):
    m = res['m']
    theta_grid = res['theta_grid']
    avg_prior = res['avg_prior']
    shrinkage = m**2 / (1 + m**2)
    Y = noise[None, :] + theta_grid[:, None]
    d_vals = delta_pi_numba(Y.ravel(), theta_grid, avg_prior).reshape(Y.shape)
    risk_sma = np.mean((d_vals - theta_grid[:, None])**2, axis=1)
    # certified bound on a fine evaluation grid (independent of the prior grid)
    theta_eval = np.linspace(-m, m, n_eval_cert)
    Y_eval = noise[None, :] + theta_eval[:, None]
    d_eval = delta_pi_numba(Y_eval.ravel(), theta_grid, avg_prior).reshape(Y_eval.shape)
    risk_eval = np.mean((d_eval - theta_eval[:, None])**2, axis=1)
    cert = certify_worst_case_risk(risk_eval, theta_eval, m)
    # certified bounds for truncated and clipped-linear on the same fine grid
    d_trunc_eval = np.clip(Y_eval, -m, m)
    risk_trunc_eval = np.mean((d_trunc_eval - theta_eval[:, None])**2, axis=1)
    cert_trunc = certify_worst_case_risk(risk_trunc_eval, theta_eval, m)
    d_lin_clip_eval = np.clip(shrinkage * Y_eval, -m, m)
    risk_lin_clip_eval = np.mean((d_lin_clip_eval - theta_eval[:, None])**2, axis=1)
    cert_lin_clip = certify_worst_case_risk(risk_lin_clip_eval, theta_eval, m)
    d_trunc = np.clip(Y, -m, m)
    risk_trunc = np.mean((d_trunc - theta_grid[:, None])**2, axis=1)
    d_lin_clip = np.clip(shrinkage * Y, -m, m)
    risk_lin_clip = np.mean((d_lin_clip - theta_grid[:, None])**2, axis=1)
    lower_bounds.append(res['bayes_risk'])
    upper_bounds.append(float(np.max(risk_sma)))
    upper_bounds_cert.append(cert['certified_upper'])
    upper_bounds_trunc.append(float(np.max(risk_trunc)))
    upper_bounds_lin_clip.append(float(np.max(risk_lin_clip)))
    upper_bounds_trunc_cert.append(cert_trunc['certified_upper'])
    upper_bounds_lin_clip_cert.append(cert_lin_clip['certified_upper'])
    m_vals.append(m)
m_vals = np.array(m_vals)
lower_bounds = np.array(lower_bounds)
upper_bounds = np.array(upper_bounds)
upper_bounds_cert = np.array(upper_bounds_cert)
upper_bounds_trunc = np.array(upper_bounds_trunc)
upper_bounds_lin_clip = np.array(upper_bounds_lin_clip)
upper_bounds_trunc_cert = np.array(upper_bounds_trunc_cert)
upper_bounds_lin_clip_cert = np.array(upper_bounds_lin_clip_cert)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(m_vals, upper_bounds_cert, 'o-', color='crimson', linewidth=1.5, markersize=5, label=r'$\sup_\theta R(d_{\pi^\epsilon}, \theta)$ (certified)')
ax.plot(m_vals, lower_bounds, 'o-', color='navy', linewidth=1.5, markersize=5, label=r'$\inf_d\, r(d, \pi^\epsilon)$')
ax.plot(m_vals, upper_bounds_trunc_cert, 'o-', color='seagreen', linewidth=1.5, markersize=5, label=r'$\sup_\theta R(d_{\mathrm{trunc}}, \theta)$ (certified)')
ax.plot(m_vals, upper_bounds_lin_clip_cert, 'o-', color='purple', linewidth=1.5, markersize=5, label=r'$\sup_\theta R(d_{\mathrm{linear,clipped}}, \theta)$ (certified)')
m_dense = np.linspace(m_vals[0], m_vals[-1], 200)
ax.plot(m_dense, m_dense**2 / (1 + m_dense**2), '--', color='dimgray', alpha=0.7, label=r'$\bar{v}_{\mathrm{linear}}(m)$')
ax.plot(m_dense, 0.8 * m_dense**2 / (1 + m_dense**2), '--', color='orange', alpha=1, label=r'$\frac{1}{1.25}\,\bar{v}_{\mathrm{linear}}(m)$')
ax.set_xlabel(r'$m$', fontsize=13)
ax.set_ylabel('Risk', fontsize=13)
ax.legend(fontsize=9, loc="lower right")
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(m_vals, upper_bounds_cert, 'o-', color='crimson', linewidth=1.5, markersize=5, label=r'$\sup_\theta R(d_{\pi^\epsilon}, \theta)$ (certified)')
ax.plot(m_vals, lower_bounds, 'o-', color='navy', linewidth=1.5, markersize=5, label=r'$\inf_d\, r(d, \pi^\epsilon)$')
m_dense = np.linspace(m_vals[0], m_vals[-1], 200)
ax.plot(m_dense, m_dense**2 / (1 + m_dense**2), '--', color='dimgray', alpha=0.7, label=r'$\bar{v}_{\mathrm{linear}}(m)$')
ax.plot(m_dense, 0.8 * m_dense**2 / (1 + m_dense**2), '--', color='orange', alpha=1, label=r'$\frac{1}{1.25}\,\bar{v}_{\mathrm{linear}}(m)$')
ax.set_xlabel(r'$m$', fontsize=13)
ax.set_ylabel('Risk', fontsize=13)
ax.legend(fontsize=9, loc="lower right")
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

In [ ]:
v_linear = m_vals**2 / (1 + m_vals**2)
mask = v_linear > 1e-10
pct_upper = np.full_like(m_vals, np.nan, dtype=float)
pct_upper[mask] = 100 * (1 - upper_bounds[mask] / v_linear[mask])
pct_upper_cert = np.full_like(m_vals, np.nan, dtype=float)
pct_upper_cert[mask] = 100 * (1 - upper_bounds_cert[mask] / v_linear[mask])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(m_vals, pct_upper_cert, 'o-', color='crimson', linewidth=1.5, markersize=5, label=r'$\sup_\theta R(d_{\pi^\epsilon}, \theta)$ (certified)')
ax.axhline(20, linestyle='--', color='orange', alpha=1, label=r'$20\%$ improvement')
ax.set_xlabel(r'$m$', fontsize=13)
ax.set_ylabel('Improvement over linear minimax (%)', fontsize=13)
ax.legend(fontsize=10, loc="lower right")
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## Figure 5 - Estimator comparison

In [ ]:
m_shape = [1, 2, 4]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, m in zip(axes, m_shape):
    res = next(r for r in full_results_all if np.isclose(r['m'], m))
    y_grid = np.linspace(-3 * m, 3 * m, 700)
    d_sma = delta_pi_numba(y_grid, res["theta_grid"], res["avg_prior"])
    shrinkage = m**2 / (1 + m**2)
    d_lin = shrinkage * y_grid
    d_trunc = np.clip(y_grid, -m, m)

    ax.plot(y_grid, d_sma, linewidth=2, color='steelblue', label=r'$d_{\pi^\epsilon}(y)$ (SMA)')
    ax.plot(y_grid, d_lin, '--', linewidth=1.5, color='red', label=r'$d_{\mathrm{linear}}(y)$')
    ax.plot(y_grid, d_trunc, ':', linewidth=1.2, color='orange', alpha=0.8, label=r'$d_{\mathrm{trunc}}(y)$')
    ax.plot(y_grid, y_grid, '-', linewidth=0.8, color='gray', alpha=0.4, label=r'$d(y)=y$')
    ax.axhline(m, color='black', linestyle=':', alpha=0.2)
    ax.axhline(-m, color='black', linestyle=':', alpha=0.2)
    ax.set_xlabel(r'$y$', fontsize=12)
    ax.set_ylabel('Estimate', fontsize=12)
    ax.set_title(rf'$m={m}$', fontsize=13)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Figure 9 - Risk function across seeds ($m=2$)


In [ ]:
def linear_minimax_risk(m):
    return m**2 / (1 + m**2)

def across_seed_risk_profile(m_profile, n_seeds=100, n_risk_profile=10000, n_theta=500):
    res = next(r for r in full_results_all if np.isclose(r['m'], m_profile))
    theta_grid_prof = res["theta_grid"]
    avg_prior_prof = res["avg_prior"]
    theta_dense = np.linspace(-m_profile, m_profile, n_theta)
    risk_matrix = np.empty((n_seeds, n_theta))
    for s in tqdm(range(n_seeds)):
        rng_s = np.random.default_rng(1000 + s)
        noise = rng_s.standard_normal(n_risk_profile)
        Y = noise[None, :] + theta_dense[:, None]
        d_vals = delta_pi_numba(Y.ravel(), theta_grid_prof, avg_prior_prof).reshape(Y.shape)
        risk_matrix[s] = np.mean((d_vals - theta_dense[:, None])**2, axis=1)
    return res, theta_dense, risk_matrix

def plot_risk_profile(m_profile, res, theta_dense, risk_matrix):
    risk_mean = risk_matrix.mean(axis=0)
    risk_lo = np.quantile(risk_matrix, 0.025, axis=0)
    risk_hi = np.quantile(risk_matrix, 0.975, axis=0)
    n_seeds = risk_matrix.shape[0]
    fig, ax = plt.subplots(figsize=(10, 5))
    for s in range(n_seeds):
        ax.plot(theta_dense, risk_matrix[s], color='navy', alpha=0.05, linewidth=0.7)
    ax.fill_between(theta_dense, risk_lo, risk_hi, color='navy', alpha=0.20, label='95% band across seeds')
    ax.plot(theta_dense, risk_mean, color='navy', linewidth=2, label=r'mean $R(d_{\pi^\varepsilon}, \theta)$')
    ax.axhline(res["bayes_risk"], color='crimson', linestyle='--', alpha=0.8, label=rf'$\inf_d r(d, \pi^\varepsilon) = {res["bayes_risk"]:.4f}$')
    ax.axhline(linear_minimax_risk(m_profile), color='dimgray', linestyle='--', alpha=0.7, label=rf'$\bar{{v}}_{{\mathrm{{linear}}}}({m_profile}) = {linear_minimax_risk(m_profile):.3f}$')
    ax.axhline(1, color='gray', linestyle=':', alpha=0.4, label=r'$R(y, \theta) = 1$')
    ax.set_xlabel(r'$\theta$', fontsize=13)
    ax.set_ylabel('Risk', fontsize=13)
    ax.legend(fontsize=10, loc="lower right")
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.show()

In [ ]:
res2, theta_dense2, risk_matrix2 = across_seed_risk_profile(2, n_seeds=100, n_risk_profile=10000)
plot_risk_profile(2, res2, theta_dense2, risk_matrix2)

## Figure 14 (Appendix E) - Theoretical vs. empirical Lipschitz constants

In [ ]:
m_lip = [1, 1.4, 2, 2.4, 3, 4]
lip_rows = []

for m in m_lip:
    res = next(r for r in full_results_all if np.isclose(r['m'], m))
    theta_eval = np.linspace(-m, m, 301)
    risk_vals = evaluate_risk_profile(
        theta_eval=theta_eval,
        theta_grid=res["theta_grid"],
        avg_prior=res["avg_prior"],
        n_risk_mc=4000,
        seed=123)
    emp = empirical_lipschitz_from_grid(theta_eval, risk_vals)
    loc = local_lipschitz_from_grid(theta_eval, risk_vals)
    L_theory = lipschitz_constant(m)
    lip_rows.append({
        "m": m,
        "L_theory": L_theory,
        "L_emp_grid": emp["L_emp"],
        "L_emp_local": loc["L_local"],
        "ratio_grid_to_theory": emp["L_emp"] / L_theory,
        "ratio_local_to_theory": loc["L_local"] / L_theory,
        "theta_i": emp["theta_i"],
        "theta_j": emp["theta_j"]})

lip_df = pd.DataFrame(lip_rows)
print(lip_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(lip_df["m"], lip_df["L_theory"], 'o--', linewidth=2, color='dimgray', label=r'Theoretical $L(m)$')
axes[0].plot(lip_df["m"], lip_df["L_emp_grid"], 'o-', linewidth=2, color='navy', label=r'Empirical $\widehat{L}_{\mathrm{grid}}(m)$')
axes[0].set_xlabel(r'$m$', fontsize=13)
axes[0].set_ylabel('Lipschitz constant', fontsize=13)
axes[0].legend(fontsize=10)
axes[1].plot(lip_df["m"], lip_df["ratio_grid_to_theory"], 'o-', linewidth=2, color='crimson')
axes[1].axhline(1.0, color='gray', linestyle='--', alpha=0.7)
axes[1].set_xlabel(r'$m$', fontsize=13)
axes[1].set_ylabel('Ratio', fontsize=13)
plt.tight_layout()
plt.show()

## Figure 16 (Appendix E) - Comparison with the Casella & Strawderman (1981) closed form ($m \le 1$)

These small $m$ are not in `full_results_all`, the SMA is rerun here (it takes a couple of minutes).

In [ ]:
# Exact minimax risk for m <= 1 via the Casella-Strawderman (1981) two-point least-favorable prior.
def cs_true_minimax_risk(m, n=8001, half=12.0):
    y = np.linspace(m - half, m + half, n)
    phi = np.exp(-0.5 * (y - m) ** 2) / np.sqrt(2 * np.pi)
    d = m * np.tanh(m * y)
    return float(np.trapz((d - m) ** 2 * phi, y))

m_markers = np.array([0.2, 0.4, 0.6, 0.8, 1.0])
n_eval_cert = 200000
noise_m1 = np.random.default_rng(123).standard_normal(1000)

sma_lower_m1 = np.empty_like(m_markers)
sma_upper_m1 = np.empty_like(m_markers)
for k, m in enumerate(m_markers):
    res = run_minimal_experiment(m=m, epsilon=epsilon_target(m),n_samples_risk=1000, seed=123, show_progress=False)
    sma_lower_m1[k] = res['bayes_risk']
    theta_eval = np.linspace(-m, m, n_eval_cert)
    Y = noise_m1[None, :] + theta_eval[:, None]
    d_vals = delta_pi_numba(Y.ravel(), res['theta_grid'], res['avg_prior']).reshape(Y.shape)
    risk = np.mean((d_vals - theta_eval[:, None]) ** 2, axis=1)
    sma_upper_m1[k] = certify_worst_case_risk(risk, theta_eval, m)['certified_upper']

m_dense = np.linspace(m_markers[0], m_markers[-1], 200)
v_star_dense = np.array([cs_true_minimax_risk(m) for m in m_dense])
v_linear_dense = m_dense ** 2 / (1 + m_dense ** 2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(m_dense, v_star_dense, '-', color='navy', linewidth=2, label=r'True minimax $\bar{v}^*(m)$')
ax.plot(m_dense, v_linear_dense, '--', color='dimgray', alpha=0.8, label=r'$\bar{v}_{\mathrm{linear}}(m)$')
ax.plot(m_dense, 0.8 * v_linear_dense, '-', color='gray', alpha=0.7, label=r'$0.8 \bar{v}_{\mathrm{linear}}(m)$')
ax.fill_between(m_markers, sma_lower_m1, sma_upper_m1, color='crimson', alpha=0.15)
ax.plot(m_markers, sma_upper_m1, 's--', color='crimson', linewidth=1.5, markersize=6, label=r'$\sup_\theta R(d_{\pi^\epsilon},\theta)$ (upper bound)')
ax.plot(m_markers, sma_lower_m1, 'o-', color='crimson', linewidth=1.5, markersize=6, label=r'$\inf_d r(d,\pi^\epsilon)$ (lower bound)')
ax.set_xlabel(r'$m$', fontsize=13)
ax.set_ylabel('Worst-case risk', fontsize=13)
ax.legend(fontsize=9, loc='upper left')
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()